
# The age–dust degeneracy on the optical g − r color

A 5 Gyr stellar population with no dust is nearly indistinguishable
from a 1 Gyr population reddened by ``τ_diff = 0.4`` when observed in
optical broadband colors alone. This is the central degeneracy that
limits SED-fitting accuracy from optical-only photometry, and the
reason FUV/NUV (sensitive to recent star formation) or rest-frame IR
(sensitive to dust mass) bands break the ambiguity.

We compute ``g − r`` over a 2-D grid in (age, ``τ_diff``) and overplot
iso-color contours; lines of constant color reveal the orientation of
the degeneracy.

References:
- Conroy 2013, ARA&A, 51, 393 (§3)
- Worthey 1994, ApJS, 95, 107 (age/Z degeneracy origin)


In [ ]:
import os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"  # suppress XLA/PjRt C++ INFO+WARNING logs

import warnings

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

import tengri
from tengri.analysis.plotting import setup_style

setup_style()
warnings.filterwarnings("ignore", message=".*BakedInBackend.*")

obs = tengri.Observation(photometry=tengri.Photometry.from_names(["sdss_g", "sdss_r"]))
model = tengri.SEDModel.build(
    tengri.load_ssp(),
    observation=obs,
    sfh={
        "type": "tsnorm",
        "*": tengri.FIXED,
        "peak_lbt_gyr": tengri.Uniform(0.1, 13.0),
        "width_gyr": 0.3,
        "log_total_mass": 10.0,
        "skew": 0.0,
        "trunc": 13.5,
    },
    dust={
        "type": "two_component",
        "*": tengri.FIXED,
        "tau_diff": tengri.Uniform(0.0, 2.0),
        "tau_bc": 0.3,
        "slope": -0.7,
    },
    redshift=tengri.Fixed(0.01),
)
baseline = dict(model.spec.sample(jax.random.PRNGKey(0)))

age_grid = np.geomspace(0.5, 12.0, 30)
tau_grid = np.linspace(0.0, 2.0, 28)
g_minus_r = np.empty((tau_grid.size, age_grid.size))

for i, tau in enumerate(tau_grid):
    for j, age in enumerate(age_grid):
        params = {
            **baseline,
            "sfh_tsnorm_peak_lbt_gyr": jnp.float64(age),
            "dust_tau_diff": jnp.float64(tau),
        }
        flux = np.asarray(model.predict_photometry(params))
        # F_nu -> AB mag, color = -2.5 log10(g/r)
        g_minus_r[i, j] = -2.5 * np.log10(flux[0] / flux[1])

fig, ax = plt.subplots(figsize=(6.8, 4.8))
mesh = ax.pcolormesh(
    age_grid, tau_grid, g_minus_r, cmap="RdYlBu_r", vmin=0.2, vmax=1.6, shading="auto"
)
levels = np.arange(0.4, 1.6, 0.1)
cs = ax.contour(
    age_grid, tau_grid, g_minus_r, levels=levels, colors="0.15", linewidths=0.6, alpha=0.8
)
ax.clabel(cs, fmt="%.1f", fontsize=7, inline=True, inline_spacing=2)
ax.set_xscale("log")
ax.set_xlabel(r"Stellar burst age [Gyr]")
ax.set_ylabel(r"Diffuse dust optical depth $\tau_{\rm diff}$  [mag]")
cbar = fig.colorbar(mesh, ax=ax, pad=0.01)
cbar.set_label(r"AB color  $g - r$  [mag]")
ax.text(
    0.05,
    0.9,
    "iso-colour lines trace the\nage–dust degeneracy",
    transform=ax.transAxes,
    fontsize=8,
    color="0.15",
    bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="0.7", lw=0.5),
)

fig.tight_layout()
plt.savefig("plot_usecase_age_dust_2d.png", dpi=150, bbox_inches="tight")